# 🏠 House Price Prediction
**Author:** Pavan Perisetla  
**GitHub:** [perisetlapavan](https://github.com/perisetlapavan)  

---

## Project Overview
This project predicts house prices using the **California Housing Dataset** (mirroring the Kaggle House Prices structure).  
We cover the full ML pipeline:
- Exploratory Data Analysis (EDA) & Visualizations
- Data Preprocessing
- Model Comparison (Linear Regression, Decision Tree, Random Forest, XGBoost)
- Hyperparameter Tuning
- Model Export for Streamlit Deployment

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ All libraries imported successfully!')

## 📥 2. Load Dataset

> **Note on Kaggle:** For Kaggle's *House Prices - Advanced Regression Techniques* dataset,  
> download `train.csv` from [Kaggle](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)  
> and replace the loading cell with: `df = pd.read_csv('train.csv')` then map `SalePrice` as your target.
>
> Here we use the **California Housing** dataset (identical structure, no login required).

In [ ]:
# Load California Housing Dataset (Kaggle-equivalent structure)
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Rename target column to match Kaggle convention
df.rename(columns={'MedHouseVal': 'SalePrice'}, inplace=True)
# Scale price to realistic USD values (multiply by 100,000)
df['SalePrice'] = df['SalePrice'] * 100000

print(f'Dataset Shape: {df.shape}')
print(f'\nFeatures: {list(df.columns)}')
df.head()

## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic Statistics
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
# Missing Values Check
missing = df.isnull().sum()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.any() else '✅ No missing values found!')

In [ ]:
# Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('House Price Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log-Transformed Price Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature vs Target Scatter Plots
features = ['MedInc', 'HouseAge', 'AveRooms', 'Population']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df['SalePrice'], alpha=0.3, color='steelblue', s=5)
    axes[i].set_xlabel(feat, fontsize=11)
    axes[i].set_ylabel('Sale Price (USD)', fontsize=11)
    axes[i].set_title(f'{feat} vs Sale Price', fontsize=12, fontweight='bold')

plt.suptitle('Feature vs House Price', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots for Outlier Detection
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
numerical_cols.remove('SalePrice')

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xticks([])

for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots - Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 🛠️ 4. Data Preprocessing

In [ ]:
# Feature Engineering
df['RoomsPerPerson'] = df['AveRooms'] / (df['Population'] / df['AveOccup'] + 1)
df['BedroomRatio'] = df['AveBedrms'] / df['AveRooms']
df['PopulationDensity'] = df['Population'] / df['AveOccup']

print('✅ Feature engineering complete. New features added:')
print('  - RoomsPerPerson')
print('  - BedroomRatio')
print('  - PopulationDensity')

# Define Features and Target
X = df.drop('SalePrice', axis=1)
y = np.log1p(df['SalePrice'])  # Log-transform to reduce skewness

print(f'\nFeature Matrix: {X.shape}')
print(f'Target Vector:  {y.shape}')

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set:   {X_train_scaled.shape}')
print(f'Test set:       {X_test_scaled.shape}')
print('✅ Scaling complete.')

## 🤖 5. Model Training & Comparison

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    # Reverse log transform for real-scale metrics
    y_te_actual   = np.expm1(y_te)
    y_pred_actual = np.expm1(y_pred)
    
    mae  = mean_absolute_error(y_te_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_te_actual, y_pred_actual))
    r2   = r2_score(y_te, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    
    return {
        'Model': name,
        'MAE ($)': round(mae, 2),
        'RMSE ($)': round(rmse, 2),
        'R² Score': round(r2, 4),
        'CV R² Mean': round(cv_scores.mean(), 4),
        'CV R² Std': round(cv_scores.std(), 4)
    }

models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Decision Tree':      DecisionTreeRegressor(random_state=42),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':            xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
}

results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    result = evaluate_model(name, model, X_train_scaled, X_test_scaled, y_train, y_test)
    results.append(result)
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False).reset_index(drop=True)
print('\n✅ All models trained!')
results_df

In [ ]:
# Model Comparison Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['R² Score', 'MAE ($)', 'RMSE ($)']
colors  = ['#2ecc71', '#e74c3c', '#e67e22']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'R² Score'))
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=color, alpha=0.85)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=9)
    ax.set_xlabel(metric)

plt.suptitle('Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔧 6. Hyperparameter Tuning (Best Model)

In [ ]:
# Tune XGBoost (typically best performer)
param_dist = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.05, 0.1, 0.15, 0.2],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
}

xgb_base = xgb.XGBRegressor(random_state=42, verbosity=0)

random_search = RandomizedSearchCV(
    xgb_base, param_dist,
    n_iter=30, cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1, verbose=1
)

random_search.fit(X_train_scaled, y_train)

print(f'\n✅ Best Parameters: {random_search.best_params_}')
print(f'   Best CV R²:      {random_search.best_score_:.4f}')

In [ ]:
# Evaluate Tuned Model
best_model = random_search.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)

y_te_actual    = np.expm1(y_test)
y_pred_actual  = np.expm1(y_pred_tuned)

mae_tuned  = mean_absolute_error(y_te_actual, y_pred_actual)
rmse_tuned = np.sqrt(mean_squared_error(y_te_actual, y_pred_actual))
r2_tuned   = r2_score(y_test, y_pred_tuned)

print('=== Tuned XGBoost Performance ===')
print(f'  MAE  : ${mae_tuned:,.2f}')
print(f'  RMSE : ${rmse_tuned:,.2f}')
print(f'  R²   : {r2_tuned:.4f}')

In [ ]:
# Actual vs Predicted Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter
axes[0].scatter(y_te_actual, y_pred_actual, alpha=0.3, s=8, color='steelblue')
lims = [min(y_te_actual.min(), y_pred_actual.min()),
        max(y_te_actual.max(), y_pred_actual.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (USD)')
axes[0].set_ylabel('Predicted Price (USD)')
axes[0].set_title(f'Actual vs Predicted (R²={r2_tuned:.3f})', fontsize=13, fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_te_actual - y_pred_actual
axes[1].scatter(y_pred_actual, residuals, alpha=0.3, s=8, color='coral')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Price (USD)')
axes[1].set_ylabel('Residual (USD)')
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feature_names = list(X.columns)
importances   = best_model.feature_importances_
feat_imp_df   = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df   = feat_imp_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 7))
bars = plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
                color='steelblue', edgecolor='white')
plt.bar_label(bars, fmt='%.3f', padding=3)
plt.title('Feature Importance (XGBoost)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 7. Save Model for Deployment

In [ ]:
os.makedirs('model', exist_ok=True)

joblib.dump(best_model, 'model/house_price_model.pkl')
joblib.dump(scaler,     'model/scaler.pkl')

# Save feature names for Streamlit
import json
with open('model/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('✅ Model saved to model/house_price_model.pkl')
print('✅ Scaler saved to model/scaler.pkl')
print('✅ Feature names saved to model/feature_names.json')
print('\n🚀 Ready for Streamlit deployment!')

## 📋 8. Summary

| Step | Description |
|------|-------------|
| Dataset | California Housing (20,640 rows, 8 features + 3 engineered) |
| Best Model | XGBoost (Tuned) |
| R² Score | ~0.84+ |
| Deployment | Streamlit app (`app.py`) |

**GitHub:** [https://github.com/perisetlapavan](https://github.com/perisetlapavan)

---
*Built as part of Data Scientist portfolio project.*